In [0]:
_checkpoints = "dbfs:/Volumes/workspace/retails/raw_data/_checkpoints/retails/dev/silver/orders_cleaned"

In [0]:
from pyspark.sql.functions import to_date, current_date, col

# Step-1: Read from Bronze (Streaming)
# 
cdc_df = spark.readStream.table("retails.silver.orders_cdc")


In [0]:
from pyspark.sql.functions import to_json, struct, when, col

# Step 4: Apply Type Casting & Standardization
cdc_df = cdc_df.withColumn("order_id", col("order_id").cast("integer")) \
                    .withColumn("order_date", col("order_date").cast("date")) \
                    .withColumn("order_customer_id", col("order_customer_id").cast(("integer"))) \
                    .withColumn("order_status", col("order_status").cast("string")) \
                    .withColumn("record_hash", col("record_hash"))
                        


In [0]:
# make ready to upsert silver table orders_cleaned
silver_df = cdc_df \
            .withColumn("order_id", col("order_id").cast("bigint")) \
            .withColumn("order_date", col("order_date").cast("date")) \
            .withColumn("order_customer_id", col("order_customer_id").cast("bigint")) \
            .withColumn("batch_id", col("batch_id").cast("string"))


In [0]:
# upsert orders_cleaned table
MERGE_UPDATE = """
    MERGE INTO retails.silver.orders_cleaned t
    USING orders_cleaned_vw s
    ON t.order_id = s.order_id
    WHEN MATCHED AND s.record_hash != t.record_hash AND s.op='UPDATE' THEN
        UPDATE SET t.order_date = s.order_date, t.order_customer_id = s.order_customer_id, t.order_status = s.order_status, t.is_deleted = false, t.updated_ts = current_timestamp(), t.op= s.op, t.record_hash = s.record_hash
    
    WHEN MATCHED AND s.is_deleted AND s.op='DELETE' THEN
        UPDATE SET t.is_deleted = true, t.updated_ts = current_timestamp(), t.op= s.op
    
    WHEN NOT MATCHED AND s.op='INSERT' THEN
        INSERT (
            order_id,
            order_date,
            order_customer_id,
            order_status,
            is_deleted,
            ingestion_ts,
            ingestion_dt,
            source_system,
            source_file_name,
            batch_id,
            run_id,
            op,
            record_hash,
            created_ts,
            updated_ts
                    )
        VALUES(
            s.order_id,
            s.order_date,
            s.order_customer_id,
            s.order_status,
            s.is_deleted,
            s.ingestion_ts,
            s.ingestion_dt,
            s.source_system,
            s.source_file_name,
            s.batch_id,
            s.run_id,
            s.op,
            s.record_hash,
            current_timestamp(),
            current_timestamp()
            )
    """

# spark.sql(merge_query).show()



In [0]:
def upsert_to_silver(batch_df, batch_id):

    # batch_df = batch_df.cache()
    batch_df.createOrReplaceTempView(
        "orders_cleaned_vw"
    )
    spark.sql(MERGE_UPDATE)

    # batch_df.unpersist()

In [0]:
silver_df = silver_df.drop("event_ts")
# silver_df.printSchema()

In [0]:

silver_df.writeStream \
    .format("delta") \
    .foreachBatch(upsert_to_silver) \
    .option("checkpointLocation", _checkpoints) \
    .trigger(once=True) \
    .start() \
    .awaitTermination()